# Sample Ingestion Notebook
This notebook demonstrates how to use `##placeholder##` tokens in your Fabric notebooks.  
At deploy time, the deployment scripts replace every `##parameterName##` with the
corresponding value from `Config/fabric_config.json`.

### Placeholders used in this notebook
| Token | Replaced With |
|---|---|
| `##fabricLakehouseId##` | Lakehouse GUID (auto-populated after creation) |
| `##lakehouseConnString##` | SQL analytics endpoint connection string |
| `##storageAccountName##` | ADLS Gen2 storage account name |
| `##fabricWorkspaceId##` | Workspace GUID (auto-resolved from name) |
| `##logAnalyticsWorkspaceId##` | Log Analytics workspace GUID |

In [ ]:
# ---------------------------------------------------------------
# Configuration — these ##placeholders## are replaced at deploy time
# ---------------------------------------------------------------
LAKEHOUSE_ID      = "##fabricLakehouseId##"
CONN_STRING       = "##lakehouseConnString##"
STORAGE_ACCOUNT   = "##storageAccountName##"
WORKSPACE_ID      = "##fabricWorkspaceId##"

In [ ]:
# ---------------------------------------------------------------
# Read data from ADLS Gen2 via the storage account placeholder
# ---------------------------------------------------------------
source_path = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/raw/events/"

df_raw = spark.read.format("parquet").load(source_path)

print(f"Loaded {df_raw.count()} rows from {source_path}")
df_raw.printSchema()

In [ ]:
# ---------------------------------------------------------------
# Transform — simple cleaning and enrichment
# ---------------------------------------------------------------
from pyspark.sql.functions import current_timestamp, col, upper

df_cleaned = (
    df_raw
    .dropDuplicates(["event_id"])
    .withColumn("ingested_at", current_timestamp())
    .withColumn("region", upper(col("region")))
    .filter(col("event_type").isNotNull())
)

print(f"Cleaned rows: {df_cleaned.count()}")

In [ ]:
# ---------------------------------------------------------------
# Write to Lakehouse delta table using the Lakehouse ID placeholder
# ---------------------------------------------------------------
target_table = f"abfss://{LAKEHOUSE_ID}@onelake.dfs.fabric.microsoft.com/Tables/bronze_events"

df_cleaned.write.format("delta").mode("append").save(target_table)

print(f"Written to {target_table}")

In [ ]:
# ---------------------------------------------------------------
# Optional — query via SQL endpoint using connection string placeholder
# ---------------------------------------------------------------
jdbc_url = f"jdbc:sqlserver://{CONN_STRING};database=contoso_lakehouse_prod"

# Example: read back via SQL endpoint for validation
# df_check = spark.read.format("jdbc").option("url", jdbc_url).option("dbtable", "bronze_events").load()
print(f"SQL endpoint: {CONN_STRING}")
print(f"Workspace ID: {WORKSPACE_ID}")